# Evaluation, finance domain (Kaggle): one source per run

Finance has four sources (docs/finance_plan.md); each owner runs **their** source on **their**
Kaggle account, so the runs go in parallel: set `SOURCE` in cell 1.

| SOURCE | Owner | Data (fetched into `data/finance/<source>/`) | Metrics |
|---|---|---|---|
| `bank` | A | CFPB complaints (real) + AgamiAI statements (population) | single-question + **multi-question** |
| `kyc` | B | Gretel documents (labelled) + NIST tax forms (scans) | single-question + end to end |
| `corporate` | C | Enron emails + TAT-DQA pages (real) + Kleister reports (scans) | single-question + end to end |
| `cheques` | C | IDRBT handwritten cheques (real scans) | end to end (handwriting OCR) |

**Before running:** GPU **T4 x2**, Internet **On**, secret **GEMINI_API_KEY** attached. For
`cheques`, also add your private Kaggle dataset with the IDRBT zip as an input and set
`CHEQUES_ZIP`. Then **Save Version -> Save & Run All (Commit)** and close Kaggle; the version's
**Output** holds `eval_results_finance_<source>_<scale>.zip`.

Steps a source does not have are skipped (`cheques` has only scans; only `bank` has attack
sessions). `SCALE = "quick"` checks the set-up in ~10 minutes; `"2h"` is the real run.

In [ ]:
# 1 Setup (run at the start of every session)
SOURCE = "bank"   # bank | kyc | corporate | cheques (your source)
CHEQUES_ZIP = ""  # cheques only: path of the IDRBT zip under /kaggle/input
SCALE = "2h"   # "2h": the real run, ~2 h per source; "quick": ~10 min check
MODEL = "llama3:8b"                       # answers, both pipelines (as the dashboard)
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"   # Agents 1-3 (fallback order)
OCR_MODEL = "qwen3-vl:8b-instruct"        # reads the scans (end-to-end check)
BRANCH = "finance"

import os, subprocess, sys
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"      # models stay out of /kaggle/working
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.install_ollama_and_model(OCR_MODEL)
colab_cache.start_ollama()

from kaggle_secrets import UserSecretsClient
try:
    key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets), tick it as "
                       "attached to this notebook, and run this cell again.") from exc
WORK = W / "eval" / f"finance_{SOURCE}_{SCALE}"
os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
    "SANITISE_GEMINI_API_KEY": key, "GEMINI_API_KEYS": key,
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
RUN = f"python tools/eval_suite/run.py {{step}} --domain finance --source {SOURCE} --work {WORK} --scale {SCALE}"
def run(step, extra=""):
    colab_cache.start_ollama()  # back up if it stopped (e.g. a cell was interrupted)
    cmd = RUN.format(step=step) + " " + extra
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)
print(f"ready: answers {MODEL} | Agents 1-3 {GEMINI_MODEL} ({len(key.split(','))} key(s)) | "
      f"OCR {OCR_MODEL} | results in {WORK}")

In [ ]:
# 2 Data: the source's folder is in the repo (each owner commits theirs); cheques are fetched
# here from your private Kaggle dataset, since the images are never committed
if SOURCE == "cheques":
    !python tools/finance/cheques.py --zip "{CHEQUES_ZIP}"
run("data")

In [ ]:
# 3 End to end: the source's scans through OCR (labels from the dataset's fields)
run("e2e-prep")

In [ ]:
# 4 Build: the finance agents over the source's documents (resumes)
run("build")

In [ ]:
# 5 Single-question attacks and utility
run("single")

In [ ]:
# 6 Multi-question attacks (bank only; skipped otherwise)
run("multi")

In [ ]:
# 7 End to end: the scans' questions
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_finance_{SOURCE}{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/utility.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"finance_{SOURCE}/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")